# Closure and Small-Algebra Diagnostics

This notebook probes the runtime-only algebra diagnostics on a few small hand-built families.

The point is not to discover a rich Lie algebra. It is to see where the current polynomial family semantics produce exact, interpretable closure diagnostics and where conditioning or redundancy starts to dominate.


In [ ]:
import numpy as np

from pdelie import GeneratorFamily
from pdelie.symmetry import diagnose_generator_family_closure, render_generator_family


In [ ]:
def translation_affine_xi_basis_spec():
    return {
        "variables": ["t", "x", "u"],
        "component_names": ["xi"],
        "basis_terms": [
            {"label": "1", "powers": [0, 0, 0]},
            {"label": "t", "powers": [1, 0, 0]},
            {"label": "x", "powers": [0, 1, 0]},
            {"label": "u", "powers": [0, 0, 1]},
        ],
        "component_ordering": ["xi"],
        "term_ordering": ["1", "t", "x", "u"],
        "layout": "component_major",
    }


def make_family(coefficients):
    return GeneratorFamily(
        parameterization="polynomial_translation_affine",
        coefficients=np.asarray(coefficients, dtype=float),
        basis_spec=translation_affine_xi_basis_spec(),
        normalization="runtime_fixture",
        diagnostics={},
    )


def summarize_closure(report):
    return {
        "computation_mode": report["computation_mode"],
        "family_rank": report["family_rank"],
        "closure_summary": report["closure"]["summary"],
        "antisymmetry_summary": report["antisymmetry"]["summary"],
        "jacobi_summary": report["jacobi"]["summary"],
        "structure_constant_shape": np.asarray(report["structure_constants"]["tensor"]).shape,
        "conditioning": report["conditioning"],
    }


In [ ]:
families = {
    "single_translation": make_family([[1.0, 0.0, 0.0, 0.0]]),
    "translation_plus_x_dx": make_family([[1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0]]),
    "nearly_dependent_pair": make_family([[1.0, 0.0, 0.0, 0.0], [1.0 + 1e-8, 0.0, 0.0, 0.0]]),
}


In [ ]:
for label, family in families.items():
    report = diagnose_generator_family_closure(family)
    print(f"=== {label} ===")
    for line in render_generator_family(family):
        print("  ", line)
    print(summarize_closure(report))
    print()


## What to look for

- the single-generator translation family should behave trivially and cleanly
- the `\u2202x` / `x\u2202x` pair should give an interpretable affine-style closure pattern
- the nearly dependent pair is useful for seeing conditioning effects without changing the nominal family meaning very much
